# Kaggle / Colab runner: stress-detection-dreaddit

This notebook runs every experiment of the project on a free cloud GPU (Kaggle T4 or Colab) and packages
the results. All numbers in the report were produced by running these cells.

**What it does, cell by cell**
1. Clone (or update) the GitHub repository.
2. Install the extra libraries and check the GPU.
3. Read the HuggingFace token (needed for the gated MentalBERT/MentalRoBERTa models).
4. Download Dreaddit and build the shared train/val/test split.
5. Experiments (each cell can be run on its own): LSTM baselines, transformer encoders (RQ1),
   LLM zero-/few-shot (RQ2), LIME + error analysis (RQ3).
6. Summarise all runs and zip `results/` and `figures/` for download.

**Kaggle settings (right panel):** Accelerator → *GPU T4 x1* (or x2), Internet → *On* (needs a phone-verified account).
**Secrets (Add-ons → Secrets):** `HF_TOKEN` = your HuggingFace read token (after accepting the MentalBERT terms).

At the end, download `outputs.zip` from the Output panel, unzip it into your local clone, and commit
`results/` + `figures/` on your branch (files containing post text are git-ignored automatically).

In [ ]:
# Clone the project repository into the working directory (Kaggle: /kaggle/working, Colab: /content).
# Re-running the cell pulls the latest commits instead of cloning again.
REPO_URL = "https://github.com/Iamdevsuyash/stress-detection-dreaddit.git"
BRANCH = "main"
import os
BASE = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
%cd $BASE
if os.path.exists("repo"):
    !git -C repo pull -q        # safe to re-run: updates instead of cloning again
else:
    !git clone -q -b $BRANCH $REPO_URL repo
%cd $BASE/repo
!git log --oneline -1           # print the commit being run (for reproducibility)

In [ ]:
# Install the libraries that Kaggle/Colab do not pre-install:
#   lime (explanations), tabulate (markdown summary table), bitsandbytes (4-bit loading of the 7B LLM).
# Then print the library versions and whether a GPU is visible.
!pip install -q lime tabulate bitsandbytes
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
# MentalBERT ships pytorch_model.bin -> needs torch >= 2.6. If older, run: !pip install -q -U "torch>=2.6"

In [ ]:
# Read the HuggingFace token from the platform's secret store and expose it as HF_TOKEN.
# The token is needed to download gated models (MentalBERT, MentalRoBERTa) after accepting their terms.
import os
try:                                    # Kaggle
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:                                # Colab
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: gated models (MentalBERT/MentalRoBERTa, Llama) will fail to download.")

In [ ]:
# Download Dreaddit and build the shared split (train 2,555 / val 283 / test 715), then print
# split sizes, % stress and segments per subreddit. Every later script reads this same split.
!python -m src.data --stats

## Experiments

Each cell below runs one script from `scripts/`. They can be run independently, except the explainability
cell, which needs the seed-13 checkpoints that the encoder cell saves. Approximate times on one T4:
LSTM ~10 min, encoders ~30 min, LLM ~50 min, explainability ~20 min.

In [ ]:
# LSTM baselines: BiLSTM + GloVe and BiLSTM with random embeddings (ablation), seeds 13/42/87 = 6 runs.
# GloVe is downloaded automatically on the first run (~860 MB); a CPU is enough for this cell.
!bash scripts/run_baselines.sh

In [ ]:
# RQ1, general vs domain-adapted encoders: BERT, MentalBERT, RoBERTa, MentalRoBERTa x seeds 13/42/87
# = 12 fine-tuning runs (~2 min each). Saves seed-13 checkpoints to models/ for the explainability cell
# and prints the McNemar / bootstrap significance tests at the end.
# Set DAPT=1 to also run the domain-adaptive pretraining stretch goal (not used in the report).
!MODELS="bert-base mentalbert roberta-base mentalroberta" DAPT=0 bash scripts/run_transformers.sh

In [ ]:
# RQ2, LLM without training: Qwen2.5-7B-Instruct in 4-bit, 0-shot (1 run) and 4-shot (seeds 13/42/87),
# each also with a decision threshold tuned on the validation split. Needs a GPU (~50 min on a T4).
!LLM="Qwen/Qwen2.5-7B-Instruct" bash scripts/run_llm.sh

In [ ]:
# RQ3, explainability + error analysis: LIME and attention for BERT vs MentalBERT (seed 13), then
# per-subreddit F1, disagreements, error lists and confusion matrices.
# Needs models/bert-base__seed13 and models/mentalbert__seed13. If the encoder cell already ran in THIS
# session, skip the two train_transformer lines (they only re-create those checkpoints; --no_record keeps
# the encoder results untouched) and run only the last line.
!python -m src.train_transformer --model bert-base --seed 13 --save --no_record
!python -m src.train_transformer --model mentalbert --seed 13 --save --no_record
!bash scripts/run_explain.sh

## Package outputs

Rebuild the summary table from every saved run and zip the results for download.

In [ ]:
# Rebuild results/summary.csv from all saved runs, then zip results/ and figures/ for download.
!python -m src.eval summarize
!zip -q -r $BASE/outputs.zip results figures
print(f"Download {BASE}/outputs.zip from the Output panel (Kaggle) or the Files pane (Colab).")